# BBQ Example

We walk through the steps of applying our method to questions from the variant of the BBQ dataset ([Parrish et al., ACL 2022](https://aclanthology.org/2022.findings-acl.165/)) introduced in ([Turpin et al., NeurIPS 2023](https://arxiv.org/abs/2305.04388)).

This notebook uses local AI agents hosted by Ollama. Make sure Ollama is running and the required models are installed:
- gemma4:e4b
- deepseek-r1:latest
- qwen3.5:9b

We evaluate three local language models:
- gemma4:e4b (used for counterfactual/intervention creation)
- deepseek-r1:latest
- qwen3.5:9b

The first part of this notebook shows the data collection steps for an example question.

The second part of this notebook shows the faithfulness estimation steps, including estimating explanation implied effects and causal concept effects. It executes these steps on the full set of counterfactual data we collected (found in ``outputs/iclr-2025/bbq``). You can run these steps to reproduce the main faithfulness plots from the paper.

### Imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import glob
import json
import os

In [ ]:
import sys

sys.path.append('../src')

In [ ]:
from causal_concept_effect_estimation.estimate_concept_effects import ConceptEffectEstimator
from explanation_implied_effect_estimation.estimate_explanation_implied_effects import ExplanationImpliedEffectEstimator
from faithfulness_estimation.estimate_faithfulness import FaithfulnessEstimator
from my_datasets.bbq import BBQDataset

In [ ]:
OUTPUT_DIR = "../outputs/bbq-example"

## Data Collection

We first walk through the steps involved in collecting the data that we later use to estimate faithfulness. This involves:
1. Concept Extraction
2. Concept Value Extraction
3. Counterfactual Question Generation
4. LLM Response Collection
5. Analyze LLM Explanations (to determine which concepts are implied to be influential)

We demonstrate steps 1-5 on the example question shown below. Then, we demonstrate how we estimate faithfulness using data collected for the larger set of BBQ questions we consider in the paper (the data for this can be found in ``outputs/iclr-2025/bbq``).

### Examine Example Question

In [ ]:
bbq_dataset = BBQDataset('bbq', '../data/bbq')

Question Metadata

In [ ]:
bbq_dataset.data[1187]

Question text

In [ ]:
print(bbq_dataset.format_prompt_basic(1187))

### Extract Concepts


We will now use gemma4:e4b as the auxiliary LLM to extract a set of concepts (i.e., distinct, high-level pieces of information) from the example question.

In this step, we also assign each concept an initial category, or higher-level "topic".

We will later map each initial category to an even more coarse-grained category (one of "identity", "behavior", "context") as a post-processing step.

Note that even though we use the model with temperature 0, the model is not deterministic -- so the concepts extracted can vary across calls to the model. This means that the concepts extracted may not match those that we used in our experiments. This is okay because there is a not a single "ground truth" concept. In fact, our method is designed to be flexible to the choice of concept set -- it assesses faithfulness with respect to the specified concept set.

In [ ]:
%%bash

python ../src/run_generate_interventions.py \
    --dataset bbq \
    --dataset_path ../data/bbq \
    --example_idxs 1187 \
    --intervention_model gemma4:e4b \
    --intervention_model_temperature 0 \
    --concept_id_only \
    --concept_id_base_prompt_name concept_id_prompt \
    --output_dir ../outputs/bbq-example/counterfactual-generation \
    --n_workers 1 \
    --verbose
    # --fresh_start # use this flag to re-run the concept extraction step; otherwise will load saved concepts from prior run


The results of this step will be in the files:
* ``../outputs/bbq-example/counterfactual-generation/example_1187/concepts.json`` (a list of concepts)
* ``../outputs/bbq-example/counterfactual-generation/example_1187/categories.json`` (a corresponding list of categories)

In [ ]:
concept_file = os.path.join(OUTPUT_DIR, "counterfactual-generation", "example_1187", "concepts.json")
categories_file = os.path.join(OUTPUT_DIR, "counterfactual-generation", "example_1187", "categories.json")
with open(concept_file, "r") as f:
    concepts = json.load(f)
with open(categories_file, "r") as f:
    categories = json.load(f)

for idx, (concept, category) in enumerate(zip(concepts, categories)):
    print(f"{idx + 1}. Concept: {concept}, Category: {category}")

### Extract Concept Values

We will now use gemma4:e4b as the auxiliary LLM to extract values for each of the concepts identified in the previous step.

For each concept, we ask the LLM to identify:
1. The concept's current value
2. A plausible alternative value for the concept. For this task, we encourage the model to choose a value that corresponds to swapping the information associated with each person in the question when applicable.

In [ ]:
%%bash

python ../src/run_generate_interventions.py \
    --dataset bbq \
    --dataset_path ../data/bbq \
    --example_idxs 1187 \
    --intervention_model gemma4:e4b \
    --intervention_model_temperature 0 \
    --concept_id_base_prompt_name concept_id_prompt \
    --concept_values_base_prompt_name concept_values_prompt \
    --concept_values_only \
    --output_dir ../outputs/bbq-example/counterfactual-generation \
    --n_workers 1 \
    --verbose
    # --fresh_start # use this flag to re-run the concept value extraction step; otherwise will load saved concept values from prior run

The results of this step will be in the file: ``../outputs/bbq-example/counterfactual-generation/example_1187/concept_settings.json``

In [ ]:
concept_file = os.path.join(OUTPUT_DIR, "counterfactual-generation", "example_1187", "concepts.json")
values_file = os.path.join(OUTPUT_DIR, "counterfactual-generation", "example_1187", "concept_settings.json")
with open(concept_file, "r") as f:
    concepts = json.load(f)
with open(values_file, "r") as f:
    values = json.load(f)

for idx, (concept, val) in enumerate(zip(concepts, values)):
    print(f"{idx + 1}. Concept: {concept}, Current value: {val['current_setting']}, New Value: {val['new_settings'][0]}")

### Generate Counterfactual Questions

We now will use gemma4:e4b to generate counterfactual questions. For each concept, we generate two new questions:
1. A "removal" based counterfactual in which the question is edited to remove the information related to the concept
2. A "replacement" based counterfactual in which the question is edited to replace the value of a concept with the alternative value identified in the previous step.

Generate Removal Based Counterfactuals

In [ ]:
%%bash

python ../src/run_generate_interventions.py \
    --dataset bbq \
    --dataset_path ../data/bbq \
    --example_idxs 1187 \
    --intervention_model gemma4:e4b \
    --intervention_model_temperature 0 \
    --concept_id_base_prompt_name concept_id_prompt \
    --concept_values_base_prompt_name concept_values_prompt \
    --counterfactual_gen_base_prompt_name counterfactual_gen_removals_prompt \
    --output_dir ../outputs/bbq-example/counterfactual-generation \
    --n_workers 1 \
    --verbose \
    --only_concept_removals
    # --fresh_start # use this flag to re-run the counterfactual generation step; otherwise will load saved counterfactuals from prior run

The results of this step will be in the directory: ``../outputs/bbq-example/counterfactual-generation/example_1187``

Each file is named ``counterfactual_XXXX.json`` where ``X=-`` indicates a concept that was removed and ``X=0`` indicates a concept that was kept the same.

Examine Removal Based Counterfactuals

In [ ]:
concept_file = os.path.join(OUTPUT_DIR, "counterfactual-generation", "example_1187", "concepts.json")
values_file = os.path.join(OUTPUT_DIR, "counterfactual-generation", "example_1187", "concept_settings.json")
with open(concept_file, "r") as f:
    concepts = json.load(f)
with open(values_file, "r") as f:
    values = json.load(f)

for intervention_file in glob.glob(os.path.join(OUTPUT_DIR, "counterfactual-generation", "example_1187", "counterfactual_*.json")):
    with open(intervention_file, "r") as f:
        intervention = json.load(f)
    if '-' not in intervention["intervention_str"]:
        continue
    intervention_idx = intervention["intervention_str"].index('-')
    concept = concepts[intervention_idx]
    val = values[intervention_idx]
    current_value = val['current_setting']
    intervention_str = f"{concept}: {current_value} -> UNKNOWN"
    print("INTERVENTION", intervention_str)
    print("COUNTERFACTUAL")
    print(intervention["parsed_counterfactual"]["edited_context"])
    print(intervention["parsed_counterfactual"]["edited_question"])
    print("A. " + intervention["parsed_counterfactual"]["edited_ans0"])
    print("B. " + intervention["parsed_counterfactual"]["edited_ans1"])
    print("C. " + intervention["parsed_counterfactual"]["edited_ans2"])
    print()

Generate Replacement Based Counterfactuals

In [ ]:
%%bash

python ../src/run_generate_interventions.py \
    --dataset bbq \
    --dataset_path ../data/bbq \
    --example_idxs 1187 \
    --intervention_model gemma4:e4b \
    --intervention_model_temperature 0 \
    --concept_id_base_prompt_name concept_id_prompt \
    --concept_values_base_prompt_name concept_values_prompt \
    --counterfactual_gen_base_prompt_name counterfactual_gen_replacements_prompt \
    --output_dir ../outputs/bbq-example/counterfactual-generation \
    --n_workers 1 \
    --verbose \
    # --fresh_start # use this flag to re-run the counterfactual generation step; otherwise will load saved counterfactuals from prior run

The results of this step will be in the directory: ``../outputs/bbq-example/counterfactual-generation/example_1187``

Each file is named ``coutnerfactual_XXXX.json`` where ``XXXX`` is counterfactual identifier string. ``X=1`` indicates a concept that was edited to a different value and ``X=0`` indicates a concept that was kept the same (there is a single `X` for each concept).

Examine Replacement Based Counterfactuals

In [ ]:
concept_file = os.path.join(OUTPUT_DIR, "counterfactual-generation", "example_1187", "concepts.json")
values_file = os.path.join(OUTPUT_DIR, "counterfactual-generation", "example_1187", "concept_settings.json")
with open(concept_file, "r") as f:
    concepts = json.load(f)
with open(values_file, "r") as f:
    values = json.load(f)

for intervention_file in glob.glob(os.path.join(OUTPUT_DIR, "counterfactual-generation", "example_1187", "counterfactual_*.json")):
    with open(intervention_file, "r") as f:
        intervention = json.load(f)
    if '1' not in intervention["intervention_str"]:
        continue
    intervention_idx = intervention["intervention_str"].index('1')
    concept = concepts[intervention_idx]
    val = values[intervention_idx]
    current_value = val['current_setting']
    new_value = val['new_settings'][0]
    intervention_str = f"{concept}: {current_value} -> {new_value}"
    print("INTERVENTION", intervention_str)
    print("COUNTERFACTUAL")
    print(intervention["parsed_counterfactual"]["edited_context"])
    print(intervention["parsed_counterfactual"]["edited_question"])
    print("A. " + intervention["parsed_counterfactual"]["edited_ans0"])
    print("B. " + intervention["parsed_counterfactual"]["edited_ans1"])
    print("C. " + intervention["parsed_counterfactual"]["edited_ans2"])
    print()

### Collect LLM Responses

We will now collect responses from the primary LLM to both the original and counterfactual questions.

As the primary LLM (i.e., the one we evaluate the faithfulness of), we will evaluate three local models:
- deepseek-r1:latest
- qwen3.5:9b

Here, we are collecting 5 model responses per question (since model outputs are not determinstic). For our experiments in the paper, we collected 50 responses. You can change how many responses to collect per question using the ``n_completions`` argument.

Note: occasionally, there may be an error in collecting the LLMs response, e.g., because of connection issues or because the LLM's response is not in the format we are expecting. If this is the case, the code below will output an error message. In addition, all errors are logged in the ``../outputs/bbq-example/model-responses/failed_examples.json`` file.

In [ ]:
%%bash

python ../src/run_collect_model_responses.py \
    --dataset bbq \
    --dataset_path ../data/bbq \
    --example_idxs 1187 \
    --language_model deepseek-r1:latest \
    --language_model_max_tokens 256 \
    --cot \
    --few_shot \
    --few_shot_prompt_name few_shot_cot_prompt \
    --n_completions 5 \
    --intervention_data_path ../outputs/bbq-example/counterfactual-generation  \
    --output_dir ../outputs/bbq-example/model-responses
    # --fresh_start # use this flag to re-run the model collection step; otherwise will load saved model responses from prior run


**Examine LLM Responses to the Original Question**

These responses will be in the directory: ``../outputs/bbq-example/model-responses/example_1187/original``

Each file is named ``response_n=i.json`` where ``i`` is the index of each of the ``n_completions`` responses.

In [ ]:
for response_file in glob.glob(os.path.join(OUTPUT_DIR, "model-responses", "example_1187", "original", "response_n=*.json")):
    with open(response_file, "r") as f:
        response = json.load(f)
    print("PROMPT")
    print(response["prompt"])
    print("\n")
    print("RESPONSE")
    print(response["response"])
    print("\n")
    print("ANSWER")
    print(response["answer"])
    print("\n\n")

**Examine LLM Responses to the Counterfactual Questions**

These responses will be in the directory: ``../outputs/bbq-example/model-responses/example_1187/counterfactual`

Each file is named ``response_counterfactual=XXXX_n=i.json`` where ``i`` is the index of each of the ``n_completions`` responses and ``XXXX`` is the counterfactual identifier (described above in the counterfactual generation step).

Here we will look at the responses to a single counterfactual -- ``-00``, where the first concept is removed -- as an example.

In [ ]:
for response_file in glob.glob(os.path.join(OUTPUT_DIR, "model-responses", "example_1187", "counterfactual", "response_counterfactual=-00_n=*.json")):
    with open(response_file, "r") as f:
        response = json.load(f)
    print("PROMPT")
    print(response["prompt"])
    print("\n")
    print("RESPONSE")
    print(response["response"])
    print("\n")
    print("ANSWER")
    print(response["answer"])
    print("\n\n")

### Analyze LLM Responses: Which Concepts does the LLM Imply are Influential?

We now use the auxiliary LLM (gemma4:e4b) to analyze the responses of the primary LLM.

Specifically, we will examine which concepts the LLM's response implied influenced its answer choice.

A couple notes on this step:
* We do not examine which concepts are mentioned in response to "removal"-based counterfactuals, since we do not expect the LLM to mention the removed concept.
* Ideally we would examine the LLM's responses to counterfactual questions in which concept values are changed; however, to reduce the computational cost of applying our method, we ended up only analyzing the responses of the LLM to the orignal questions. In a few premilinary experiments, we found that the concepts mentioned in both types of questions (original and "replacement"-based counterfactuals) were largely the same.
* Occassionally, there may be in an error in analying the LLM's response, e.g., because of connection issues to the auxiliary LLM or because the auxiliary LLM's response is not in the format we are expecting. In this case, the code will output an error message and the errors will be logged in ``../outputs/bbq-example/implied-concepts/failed_examples.json``.

In [ ]:
%%bash

python ../src/run_determine_implied_concepts.py \
    --dataset bbq \
    --dataset_path ../data/bbq \
    --example_idxs 1187 \
    --implied_concepts_model gemma4:e4b \
    --implied_concepts_base_prompt_name implied_concepts_prompt \
    --intervention_data_path ../outputs/bbq-example/counterfactual-generation  \
    --model_response_data_path  ../outputs/bbq-example/model-responses \
    --output_dir ../outputs/bbq-example/implied-concepts \
    --original_only \
    --verbose
    # --fresh_start # use this flag to re-run the response analysis step; otherwise will load saved model responses from prior run

Let's examine the outputs from this step.

In [ ]:
for implied_concept_file in glob.glob(os.path.join(OUTPUT_DIR, "implied-concepts", "example_1187", "original", "implied_concepts_*.json")):
    with open(implied_concept_file, "r") as f:
        implied_concepts = json.load(f)
    print("PROMPT")
    print(implied_concepts["prompt"])
    print("\n")
    print("RESPONSE")
    print(implied_concepts["responses"][0])
    print("\n")
    print("IMPLIED CONCEPT DECISIONS")
    print(implied_concepts["concept_decisions"][0])
    print("\n\n")

## Faithfulness Measurement

We now walk through the steps of estimating caual concept faithfulness using the data collected using steps 1-5 above.

We will do this on the full set of examples used in the paper; the data for these examples is found in ``outputs/iclr-2025/bbq``.

The key steps for this are:
1. Estimating Explanation-Implied Effects (EE)
2. Estimating Causal Concept Effects (CE)
3. Measuring Causal Concept Faithfulness (the correlation between EE and CE)

In [ ]:
IMPLIED_CONCEPTS_DIR = "../outputs/iclr-2025/bbq/implied-concepts"
INTERVENTION_DIR = "../outputs/iclr-2025/bbq/counterfactual-generation"
MODEL_RESPONSE_DIR = "../outputs/iclr-2025/bbq/model-responses"

Note: we analyze the following the 29 questions. We dropped question 367 from the analysis, since upon manual inspection of the counterfactuals, we found some clear errors.

In [ ]:
BBQ_EX_IDXS = [578, 738, 2351, 135, 176, 183, 430, 462, 726, 761, 858, 911, 913, 1058, 1175, 1187, 1250, 1462, 1675, 1714, 1768, 1844, 1876, 2056, 2079, 2209, 2356, 2414, 2476]

We perform this analysis for each of the three LLMs examined: deepseek-r1:latest, qwen3.5:9b

### deepseek-r1:latest

#### (1) Estimating Explanation-Implied Effects

First, we read in the ``concept_decisions`` from the step in which we analyzed the LLM responses to determine which concepts its explanations implied influenced its answer choice. For each question, we have a list of the decision for each concept (1 for yes, 0 for no) in the question.

In [ ]:
deepseek_ic_dir = os.path.join(IMPLIED_CONCEPTS_DIR, "gpt-3.5-few-shot-cot")

deepseek_ee_estimator = ExplanationImpliedEffectEstimator(bbq_dataset, BBQ_EX_IDXS, INTERVENTION_DIR, deepseek_ic_dir, verbose=True)
deepseek_ic_df = deepseek_ee_estimator.load_data(load_counterfactual_responses=False)
deepseek_ic_df.head()

Next, we average the ``concept_decisions`` across the 50 responses per question to get the empirical probabilites that a concept is implied as influential. In the dataframe below ``p(concept_in_explanation)`` is the explanation-implied effect.

In [ ]:
deepseek_ee_df = deepseek_ee_estimator.estimate_implied_effects(deepseek_ic_df)
deepseek_ee_df.head()

#### (2) Estimating Causal Concept Effects

First we read in the model's responses to the original and counterfactual questions.

In [ ]:
deepseek_mr_dir = os.path.join(MODEL_RESPONSE_DIR, "gpt-3.5-few-shot-cot")
deepseek_ce_estimator = ConceptEffectEstimator(bbq_dataset, BBQ_EX_IDXS, INTERVENTION_DIR, deepseek_mr_dir, verbose=True)
deepseek_response_df = deepseek_ce_estimator.load_data()

In [ ]:
deepseek_response_df.head()

Fit hiearchical Bayesian model of concept effects

In [ ]:
deepseek_samples, deepseek_cats, deepseek_treatments, deepseek_treatment_ref_classses = deepseek_ce_estimator.fit_logistic_regression_hierarchical_bayesian(deepseek_response_df)

Estimate causal effects using the samples from the posterior distribution of model parameters

In [ ]:
deepseek_cat_df, deepseek_treatment_df = deepseek_ce_estimator.get_parameter_results_from_posterior_samples(deepseek_samples, deepseek_cats, deepseek_treatments, deepseek_treatment_ref_classses, deepseek_response_df)

View the posterior mean estimates of the category-specific sigma parameters

In [ ]:
deepseek_cat_df

View the posterior mean estimates of the beta-parameters (for each concept intervention/treatment and response variable)

In [ ]:
deepseek_treatment_df.head()

#### (3) Estimating Causal Concept Faithfulness

In [ ]:
deepseek_faithfulness_estimator = FaithfulnessEstimator(deepseek_ee_df, deepseek_treatment_df)

Fit hierarchical Bayesian model regressing the explanation implied effects on the causal concept effects of each concept

In [ ]:
deepseek_faith_samples, deepseek_beta_mean, deepseek_beta_credible_interval = deepseek_faithfulness_estimator.estimate_faithfulness()

View posterior mean and 90% credible interval of beta parameter (i.e., faithfulness value)

In [ ]:
deepseek_beta_mean

In [ ]:
deepseek_beta_credible_interval

Create faithfulness plot (causal concept effect vs explanation implied effect)

In [ ]:
deepseek_faithfulness_estimator.plot_faithfulness(deepseek_faith_samples)

### qwen3.5:9b

#### (1) Estimating Explanation-Implied Effects

In [ ]:
qwen_ic_dir = os.path.join(IMPLIED_CONCEPTS_DIR, "qwen3.5:9b-few-shot-cot")

qwen_ee_estimator = ExplanationImpliedEffectEstimator(bbq_dataset, BBQ_EX_IDXS, INTERVENTION_DIR, qwen_ic_dir, verbose=True)
qwen_ic_df = qwen_ee_estimator.load_data(load_counterfactual_responses=False)
qwen_ic_df.head()

In [ ]:
qwen_ee_df = qwen_ee_estimator.estimate_implied_effects(qwen_ic_df)
qwen_ee_df.head()

#### (2) Estimating Causal Concept Effects

In [ ]:
qwen_mr_dir = os.path.join(MODEL_RESPONSE_DIR, "qwen3.5:9b-few-shot-cot")
qwen_ce_estimator = ConceptEffectEstimator(bbq_dataset, BBQ_EX_IDXS, INTERVENTION_DIR, qwen_mr_dir, verbose=True)
qwen_response_df = qwen_ce_estimator.load_data()

In [ ]:
qwen_response_df.head()

In [ ]:
qwen_samples, qwen_cats, qwen_treatments, qwen_treatment_ref_classses = qwen_ce_estimator.fit_logistic_regression_hierarchical_bayesian(qwen_response_df)

In [ ]:
qwen_cat_df, qwen_treatment_df = qwen_ce_estimator.get_parameter_results_from_posterior_samples(qwen_samples, qwen_cats, qwen_treatments, qwen_treatment_ref_classses, qwen_response_df)

In [ ]:
qwen_cat_df

In [ ]:
qwen_treatment_df.head()

#### (3) Estimating Causal Concept Faithfulness

In [ ]:
qwen_faithfulness_estimator = FaithfulnessEstimator(qwen_ee_df, qwen_treatment_df)

In [ ]:
qwen_faith_samples, qwen_beta_mean, qwen_beta_credible_interval = qwen_faithfulness_estimator.estimate_faithfulness()

In [ ]:
qwen_beta_mean

In [ ]:
qwen_beta_credible_interval

In [2]:
qwen_faithfulness_estimator.plot_faithfulness(qwen_faith_samples)

NameError: name 'qwen_faithfulness_estimator' is not defined